# Book Finder - scientific papers & books downloader

Searches legal open-access sources (**arXiv, OpenAlex, Semantic Scholar, DOAJ** for papers; **Internet Archive, Project Gutenberg** for books) and bulk-downloads them to Google Drive:

- Files -> `MyDrive/OCR Engine/raw/papers/<tag>/...` and `.../raw/books/<tag>/...`
- Tracking -> every file is SHA-256-hashed into `MyDrive/OCR Engine/config/download_tracker.db`, so **the same file is never saved twice**

**How to use:** run Cell 1 once per session, edit Cell 2, run Cell 3.

**You never need to re-upload this notebook** - Cell 1 pulls the latest code from GitHub on every run.

In [ ]:
# Cell 1 - SETUP: pull the latest code from GitHub, install deps, mount Drive.
#
# All logic lives in the GitHub repo; this cell fetches the newest version of
# it on every run, so code changes never require re-uploading this notebook.
import os, sys, subprocess

REPO_URL = "https://github.com/greencube-dev-boop/book-finder.git"
REPO_DIR = "/content/book-finder"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                os.path.join(REPO_DIR, "requirements.txt")], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

from google.colab import drive
drive.mount("/content/drive")

import finder
print("Book Finder ready - code pulled from:", REPO_URL)
print("Version:", finder.__version__)

## Cell 2 - Choose what to find

- **TAGS**: comma-separated search tags, e.g. `quantum computing, machine learning` (for papers you can also use an arXiv category like `cs.AI`)
- **MEDIA**: `papers`, `books` or `both`
- **MAX_DOWNLOADS**: how many files to try downloading **this run** (choose any number)
- **SOURCES**: `all`, or `papers`, or `books`, or comma-separated names: `arxiv, openalex, semanticscholar, doaj, archive, gutenberg`
- **MIN/MAX SIZE_MB**: files outside this range are skipped (0.01 MB = 10 KB)

In [ ]:
# Cell 2 - WHAT TO FIND (edit and re-run Cells 2+3 any time)
TAGS = "quantum computing, machine learning" #@param {type:"string"}
MEDIA = "papers" #@param ["papers", "books", "both"]
MAX_DOWNLOADS = 100 #@param {type:"integer"}
SOURCES = "all" #@param {type:"string"}
MIN_SIZE_MB = 0.01 #@param {type:"number"}
MAX_SIZE_MB = 100 #@param {type:"number"}

TAG_LIST = [t.strip() for t in TAGS.split(",") if t.strip()]
print("Tags:", TAG_LIST)
print("Media:", MEDIA, "| Max downloads:", MAX_DOWNLOADS, "| Sources:", SOURCES)

## Cell 3 - Run

Prints **found / downloaded / duplicates skipped / failed** at the end and saves a JSON run report under `OCR Engine/config/reports/`.

Re-running is always safe: URLs already in the tracker are skipped, and files whose SHA-256 already exists are not saved twice. URLs that failed 3 times are skipped in future runs.

In [ ]:
# Cell 3 - RUN: search, download into Drive, track with SHA-256, print summary
from finder.pipeline import RunConfig, run

cfg = RunConfig(
    tags=TAG_LIST,
    media=MEDIA,
    max_downloads=MAX_DOWNLOADS,
    sources=SOURCES,
    min_size_bytes=int(MIN_SIZE_MB * 1024 * 1024),
    max_size_bytes=int(MAX_SIZE_MB * 1024 * 1024),
)
summary = run(cfg)

In [ ]:
# Cell 4 (optional) - peek inside the tracker database
from finder import config as fcfg
from finder.tracker import DownloadTracker

tracker = DownloadTracker(fcfg.paths_for(fcfg.default_data_root())["db"])
print("DB:", tracker.db_path)
print("Unique files tracked:", tracker.stats())
print("Most recent downloads:")
for row in tracker.recent(15):
    print("  -", row)
tracker.close()